# 03 · Logistic regression and classification

After this notebook you can explain the sigmoid and log-loss, derive the gradient, train logistic regression from scratch in NumPy, draw its decision boundary, pick a threshold from business costs instead of defaulting to 0.5, extend it to many classes with softmax, and say what `C` does.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · Linear regression and gradient descent](02_linear_regression_and_gradient_descent.ipynb)

## Why this matters in interviews

- Logistic regression is the baseline classifier everyone expects you to try first: fast, interpretable, and its probabilities are usually decent. A logistic regression on TF-IDF is the baseline an LLM classifier must beat; model routers are often exactly this.
- Log-loss *is* cross-entropy, the same loss that trains every LLM on next-token prediction. Logits, softmax and temperature are the same maths you will meet in decoding.
- "Why 0.5?" is where ML meets product. Choosing a threshold from the cost of each mistake is a favourite follow-up.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo01` | What is generative AI, and how is it different from the machine learning that came before it? |
| `fo22` | What are logits, and what is the softmax doing? |
| `fo14` | What is temperature actually doing, mathematically? |
| `tn02` | Explain next-token prediction as a training objective. Why is it enough? |
| `po12` | How would you build a model router, and what decides when to escalate? |

**Also asked in classical-ML rounds:** why not MSE for classification? derive the log-loss gradient; is the decision boundary linear? what does `C` control? one-vs-rest vs softmax; how do you read a coefficient?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy.special import expit
from sklearn.datasets import load_breast_cancer, load_iris, make_blobs, make_classification, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, recall_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, "#eda100", "#4a3aa7"])})
BLUE_ORANGE = LinearSegmentedColormap.from_list("blue_orange", [BLUE, "#f0efec", ORANGE])   # class 0 <-> class 1

def plot_boundary(ax, proba_fn, X, y, title):
    """Shade P(class 1) over a grid, draw the 0.5 boundary in black, then the points."""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
    p = proba_fn(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=np.linspace(0, 1, 11), cmap=BLUE_ORANGE, alpha=0.45)
    ax.contour(xx, yy, p, levels=[0.5], colors="black", linewidths=1.5)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 1, ORANGE, BLUE), s=14, edgecolors="white", linewidths=0.4)
    ax.set(title=title, xlabel="x1", ylabel="x2")

## 1. From a score to a probability: the sigmoid

**In plain English:** a linear score $z = w\cdot x + b$ can be any number. The sigmoid squashes it into (0, 1) so we can read it as $P(y=1 \mid x)$. Big positive score → near 1, big negative → near 0, zero → exactly 0.5.

$$\sigma(z) = \frac{1}{1+e^{-z}} \qquad z = \log\frac{p}{1-p}\ \ \text{(the log-odds, or **logit**)}$$

So logistic regression is linear regression on the log-odds. That is why it is called "regression" although it classifies, and why its decision boundary ($z = 0$) is a straight line (a hyperplane) in feature space.

In [ ]:
def sigmoid(z):
    """Numerically stable: never calls exp() on a large positive number."""
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1 / (1 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1 + ez)
    return out

z = np.linspace(-8, 8, 401)
assert np.allclose(sigmoid(z), expit(z))                          # matches scipy's reference
assert np.allclose(sigmoid(-z), 1 - sigmoid(z))                   # symmetry
assert np.allclose(np.log(sigmoid(z) / (1 - sigmoid(z))), z)      # logit is the inverse
assert np.isfinite(sigmoid(np.array([-1000.0, 1000.0]))).all()    # no overflow at extremes

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(z, sigmoid(z), lw=2.5, color=BLUE)
ax.axhline(0.5, color=GREY, ls="--", lw=1); ax.axvline(0, color=GREY, ls="--", lw=1)
ax.annotate("z = 0  ->  p = 0.5\n(the decision boundary)", xy=(0, 0.5), xytext=(1.5, 0.2), arrowprops=dict(arrowstyle="->"))
ax.set(title="The sigmoid turns a score (logit) into a probability", xlabel="z = w·x + b (logit)", ylabel="P(y = 1 | x)")
plt.show()

## 2. The loss: log-loss (binary cross-entropy), and why not MSE

**In plain English:** for every example, pay $-\log(\text{the probability you gave to the true answer})$. Confident and right costs almost nothing; confident and wrong costs a lot.

$$L(w) = -\frac1n\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big], \qquad p_i = \sigma(w\cdot x_i + b)$$

It is the negative log-likelihood of a coin-flip (Bernoulli) model, so minimising it is maximum likelihood. Why not reuse MSE?

1. **Vanishing gradient when confidently wrong.** With a sigmoid, $\partial\,\text{MSE}/\partial z = 2(p-y)\,p(1-p)$, and the $p(1-p)$ factor goes to zero exactly when the model is most wrong. The log-loss gradient is simply $p - y$.
2. **Convexity.** Log-loss with a linear score is convex: one global minimum. MSE through a sigmoid is not.
3. **Probabilities.** Log-loss is a *proper scoring rule*: it is minimised by reporting your true belief, which is why logistic regression's probabilities tend to be well calibrated.

In [ ]:
zs = np.linspace(-8, 8, 400)
p = sigmoid(zs)                          # the true label is y = 1 throughout
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(p, -np.log(p), lw=2, label="log-loss  -log p")
axes[0].plot(p, (1 - p) ** 2, lw=2, label="squared error  (1-p)²")
axes[0].set(title="Loss when the truth is y = 1", xlabel="predicted p", ylabel="loss", ylim=(0, 5)); axes[0].legend()
axes[1].plot(zs, p - 1, lw=2, label="log-loss gradient  p - y")
axes[1].plot(zs, 2 * (p - 1) * p * (1 - p), lw=2, label="MSE gradient  2(p-y)p(1-p)")
axes[1].axvspan(-8, -4, color=GREY, alpha=0.15)
axes[1].text(-7.8, -0.55, "confidently wrong:\nMSE barely learns", fontsize=9)
axes[1].set(title="Gradient w.r.t. the logit z (y = 1)", xlabel="logit z", ylabel="dLoss/dz"); axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

z_wrong = -6.0                           # the model is very sure of the wrong class
p_wrong = expit(z_wrong)
print(f"at z = -6: log-loss gradient {p_wrong - 1:.3f}, MSE gradient {2 * (p_wrong - 1) * p_wrong * (1 - p_wrong):.4f}")
assert abs(p_wrong - 1) > 0.99 and abs(2 * (p_wrong - 1) * p_wrong * (1 - p_wrong)) < 0.01

**The LLM connection (`tn02`, `fo01`).** Next-token prediction uses exactly this loss: $-\log P(\text{the actual next token})$, averaged over positions, with a softmax over the vocabulary instead of a sigmoid over two classes (section 6). Perplexity is just $e^{\text{mean loss}}$. Classical ML *picks* a label from a fixed set; an LLM is this same classifier applied over and over, feeding its own picks back in.

## 3. Logistic regression from scratch with gradient descent

The gradient has the same shape as linear regression's; only $p$ changed:

$$\nabla_w L = \frac1n X^\top (p - y), \qquad \frac{\partial L}{\partial b} = \frac1n\sum_i (p_i - y_i)$$

scikit-learn's `LogisticRegression` also adds an L2 penalty by default. Its objective is $C\sum_i \ell_i + \tfrac12\lVert w\rVert^2$, which equals our mean loss plus $\tfrac{1}{2Cn}\lVert w\rVert^2$. Add that term and we can check our answer against theirs exactly.

In [ ]:
def log_loss_np(y, p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def fit_logreg(X, y, lr=0.5, epochs=3000, C=None):
    """Batch GD on mean log-loss, plus sklearn-style L2 if C is given. Returns w, b and the loss per epoch."""
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    lam = 0.0 if C is None else 1.0 / (C * n)
    history = []
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        history.append(log_loss_np(y, p) + 0.5 * lam * w @ w)
        w -= lr * (X.T @ (p - y) / n + lam * w)
        b -= lr * np.mean(p - y)
    return w, b, np.array(history)

Xb, yb = make_blobs(n_samples=300, centers=[[-1, -1], [1.2, 1.0]], cluster_std=1.1, random_state=0)
Xb = StandardScaler().fit_transform(Xb)
w, b, hist = fit_logreg(Xb, yb, C=1.0)
sk = LogisticRegression(C=1.0).fit(Xb, yb)
print(f"ours   : w = {w.round(4)}, b = {b:.4f}")
print(f"sklearn: w = {sk.coef_[0].round(4)}, b = {sk.intercept_[0]:.4f}")
assert np.allclose(w, sk.coef_[0], atol=1e-3) and abs(b - sk.intercept_[0]) < 1e-3
assert np.isclose(log_loss_np(yb, sigmoid(Xb @ w + b)), log_loss(yb, sk.predict_proba(Xb)[:, 1]), atol=1e-4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist, lw=2, color=BLUE)
axes[0].set(title="Training loss (log-loss + L2)", xlabel="epoch", ylabel="loss", xscale="log")
plot_boundary(axes[1], lambda Z: sigmoid(Z @ w + b), Xb, yb, f"Blobs: a straight boundary (accuracy {sk.score(Xb, yb):.2f})")
plt.tight_layout(); plt.show()

The shading is $P(y=1)$: it fades through 0.5 at the black line. The boundary is where $w\cdot x + b = 0$, a straight line. Points in the overlap get probabilities near 0.5, which is the model being honest about its uncertainty.

## 4. When a straight line is not enough: `make_moons`

The model is linear *in its features*, so give it better features. `PolynomialFeatures(3)` adds $x_1^2, x_1x_2, x_2^3, \dots$ and the boundary can curve.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.25, random_state=0)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=0)
linear = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xm_tr, ym_tr)
cubic = make_pipeline(PolynomialFeatures(3), StandardScaler(), LogisticRegression(C=10, max_iter=2000)).fit(Xm_tr, ym_tr)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, model, name in [(axes[0], linear, "raw features"), (axes[1], cubic, "cubic features")]:
    plot_boundary(ax, lambda Z, m=model: m.predict_proba(Z)[:, 1], Xm, ym, f"{name}: test accuracy {model.score(Xm_te, ym_te):.2f}")
plt.tight_layout(); plt.show()
assert cubic.score(Xm_te, ym_te) > linear.score(Xm_te, ym_te) + 0.05

## 5. `predict_proba`, and why 0.5 is a choice

`predict()` is just `predict_proba() >= 0.5`. The 0.5 is only right when both mistakes cost the same. If a miss (false negative) costs $C_{FN}$ and a false alarm costs $C_{FP}$, predict positive when the expected cost of saying "no" is higher than that of saying "yes":

$$p\cdot C_{FN} > (1-p)\cdot C_{FP} \iff p > t^* = \frac{C_{FP}}{C_{FP}+C_{FN}}$$

A miss 10× as costly as a false alarm gives $t^* = 1/11 \approx 0.09$. The formula assumes **calibrated** probabilities, so we test it where that holds: 20,000 rows drawn from a true logistic model, fit on half, costs counted on the other half.

In [ ]:
COST_FN, COST_FP = 10, 1
t_star = COST_FP / (COST_FP + COST_FN)
Xs = rng.normal(size=(20_000, 3))
ys = (rng.random(20_000) < expit(Xs @ np.array([1.5, -1.0, 0.5]) - 1.0)).astype(int)   # labels from a real logistic model
model_s = LogisticRegression().fit(Xs[:10_000], ys[:10_000])
p_val = model_s.predict_proba(Xs[10_000:])[:, 1]
y_val = ys[10_000:]

def total_cost(y_true, p, t):
    pred = p >= t
    return COST_FN * np.sum((y_true == 1) & ~pred) + COST_FP * np.sum((y_true == 0) & pred)

thresholds = np.linspace(0.01, 0.99, 99)
costs = np.array([total_cost(y_val, p_val, t) for t in thresholds])
t_best = thresholds[costs.argmin()]
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(thresholds, costs, lw=2, color=BLUE)
ax.axvline(0.5, color=GREY, ls="--", label=f"default 0.5 (cost {total_cost(y_val, p_val, 0.5)})")
ax.axvline(t_star, color=ORANGE, lw=2, label=f"theory C_FP/(C_FP+C_FN) = {t_star:.2f}")
ax.plot(t_best, costs.min(), "o", color="black", label=f"empirical best {t_best:.2f} (cost {costs.min()})")
ax.set(title="Total cost vs threshold (a miss costs 10x a false alarm)", xlabel="threshold", ylabel="total cost"); ax.legend()
plt.show()
assert abs(t_best - t_star) < 0.05 and costs.min() < 0.8 * total_cost(y_val, p_val, 0.5)

The empirical minimum lands on the formula, and the default 0.5 costs far more. Now a real, smaller dataset: breast cancer, where the positive class is **malignant**. We choose the threshold on *out-of-fold* probabilities (`cross_val_predict`), so no row is scored by a model that saw it.

In [ ]:
data = load_breast_cancer()
Xc, yc = data.data, 1 - data.target                       # 1 = malignant: the class we must not miss
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
p_oof = cross_val_predict(clf, Xc, yc, cv=5, method="predict_proba")[:, 1]
costs_c = np.array([total_cost(yc, p_oof, t) for t in thresholds])
t_c = thresholds[costs_c.argmin()]
for label, t in [("default", 0.5), ("chosen", t_c)]:
    pred = p_oof >= t
    fn, fp = np.sum((yc == 1) & ~pred), np.sum((yc == 0) & pred)
    print(f"{label:>7} threshold {t:.2f}: missed cancers {fn:2d}, false alarms {fp:2d}, cost {COST_FN * fn + COST_FP * fp}")
assert costs_c.min() < total_cost(yc, p_oof, 0.5)

Same model, same probabilities. Only the threshold moved, and missed cancers dropped while false alarms rose, which is the trade the costs asked for. On small real data the cost curve is jumpy and the probabilities are not perfectly calibrated, so pick the threshold on validation data and use the formula as a sanity check. (Choosing *and* reporting on the same out-of-fold scores is slightly optimistic; for the final number, use an untouched test set.)

**Model routers (`po12`)** are the same decision: "send this request to the expensive model if P(hard) > t", where t comes from the cost of a bad cheap answer versus the cost of an unnecessary escalation.

## 6. Many classes: softmax on iris

**In plain English:** compute one score (logit) per class, then turn the scores into probabilities that sum to 1. The biggest score gets the biggest share.

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}, \qquad \text{loss} = -\log p_{\text{true class}}$$

Softmax is **shift-invariant** (adding a constant to every logit changes nothing), so implementations subtract the max logit first to avoid overflow (`fo22`). With two classes, softmax reduces to the sigmoid.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)          # shift-invariance makes this free and overflow-proof
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

logits = np.array([[1000.0, 999.0, 990.0]])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(logits) / np.exp(logits).sum()
print("naive softmax of [1000, 999, 990]:", naive.round(4), "   stable:", softmax(logits).round(4))
assert np.isnan(naive).all() and np.isclose(softmax(logits).sum(), 1)
assert np.allclose(softmax(logits), softmax(logits - 1000))            # shift-invariance
two = np.array([[2.0, 0.0]])
assert np.isclose(softmax(two)[0, 0], expit(2.0))                        # 2 classes: softmax == sigmoid

In [ ]:
iris = load_iris()
Xi = StandardScaler().fit_transform(iris.data)
yi = iris.target

def fit_softmax(X, y, K, lr=0.5, epochs=5000, C=1.0):
    """Multinomial logistic regression by batch GD, with sklearn-style L2 (lam = 1 / (C n))."""
    n, d = X.shape
    W, bvec, Y, lam = np.zeros((d, K)), np.zeros(K), np.eye(K)[y], 1.0 / (C * n)
    for _ in range(epochs):
        P = softmax(X @ W + bvec)
        W -= lr * (X.T @ (P - Y) / n + lam * W)      # same (p - y) form, one column per class
        bvec -= lr * (P - Y).mean(axis=0)
    return W, bvec

W, bvec = fit_softmax(Xi, yi, K=3)
P_ours = softmax(Xi @ W + bvec)
sk_multi = LogisticRegression(C=1.0, max_iter=1000).fit(Xi, yi)
P_sk = sk_multi.predict_proba(Xi)
print(f"accuracy ours {accuracy_score(yi, P_ours.argmax(1)):.3f}, sklearn {sk_multi.score(Xi, yi):.3f}")
print(f"largest probability difference vs sklearn: {np.abs(P_ours - P_sk).max():.5f}")
assert np.abs(P_ours - P_sk).max() < 0.005 and np.allclose(P_ours.sum(axis=1), 1)

In [ ]:
Xi2 = iris.data[:, 2:]                                          # petal length, petal width: plottable
model2 = LogisticRegression(max_iter=1000).fit(Xi2, yi)
xx, yy = np.meshgrid(np.linspace(0.5, 7.5, 300), np.linspace(-0.2, 2.8, 300))
regions = model2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 4))
ax.contourf(xx, yy, regions, levels=[-0.5, 0.5, 1.5, 2.5], colors=[BLUE, ORANGE, AQUA], alpha=0.2)
for k, color in enumerate([BLUE, ORANGE, AQUA]):
    ax.scatter(*Xi2[yi == k].T, s=16, color=color, edgecolors="white", linewidths=0.4, label=iris.target_names[k])
ax.set(title=f"Softmax regression on iris: three linear regions (accuracy {model2.score(Xi2, yi):.2f})",
       xlabel="petal length (cm)", ylabel="petal width (cm)"); ax.legend(); plt.show()

### Temperature: dividing the logits before the softmax (`fo14`)

LLM sampling uses exactly this: $p = \text{softmax}(z / T)$. $T < 1$ sharpens towards the top class (greedy as $T\to0$); $T > 1$ flattens towards uniform. Here are the three class probabilities for one iris flower that sits between two classes.

In [ ]:
row = np.argmin(np.abs(P_ours[:, 1] - 0.6))                     # a flower the model is unsure about
z_row = (Xi[row] @ W + bvec)[None, :]
temps = [0.25, 0.5, 1.0, 2.0, 5.0]
probs_T = np.vstack([softmax(z_row / T)[0] for T in temps])

fig, ax = plt.subplots(figsize=(8, 3.4))
width = 0.26
for k, color in enumerate([BLUE, ORANGE, AQUA]):
    ax.bar(np.arange(len(temps)) + (k - 1) * width, probs_T[:, k], width=width, color=color, label=iris.target_names[k])
ax.set_xticks(np.arange(len(temps)), [f"T = {T}" for T in temps])
ax.set(title=f"softmax(z / T) for one flower, logits z = {z_row[0].round(2)}", ylabel="probability", ylim=(0, 1)); ax.legend()
plt.show()
assert probs_T[0].max() > probs_T[2].max() > probs_T[-1].max()        # colder = more peaked
assert np.abs(probs_T[-1] - 1 / 3).max() < np.abs(probs_T[2] - 1 / 3).max()   # hotter = closer to uniform

Temperature never changes *which* class is on top, only how peaked the distribution is. That is why temperature 0 (greedy) is deterministic and high temperatures give more varied LLM output. More in [decoding: temperature, top-k, top-p](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb).

## 7. Regularisation strength `C`

`C` is the **inverse** of the regularisation strength: small `C` means a strong penalty on large weights, so the boundary is simpler and probabilities stay near 0.5. Large `C` lets weights grow, so the boundary gets sharper and wigglier. Same moons data, degree-5 features:

In [ ]:
Cs = [0.01, 1, 1000]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
norms = []
for ax, C in zip(axes, Cs):
    m = make_pipeline(PolynomialFeatures(5), StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(Xm_tr, ym_tr)
    norms.append(np.linalg.norm(m[-1].coef_))
    plot_boundary(ax, lambda Z, m=m: m.predict_proba(Z)[:, 1], Xm, ym,
                  f"C = {C}: train {m.score(Xm_tr, ym_tr):.2f} / test {m.score(Xm_te, ym_te):.2f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
fig.suptitle("Small C = strong penalty (soft, simple); large C = weak penalty (sharp, flexible)", y=1.02); plt.show()
print("weight norm |w| for C = 0.01, 1, 1000:", np.round(norms, 2))
assert norms[0] < norms[1] < norms[2]

Two penalty shapes, and how to ask for them in scikit-learn ≥ 1.8 (older code wrote `penalty="l1"`; that argument is now deprecated):

| Penalty | scikit-learn | Effect |
|---|---|---|
| L2 (default) | `LogisticRegression(C=...)` | shrinks all weights smoothly |
| L1 | `LogisticRegression(C=..., l1_ratio=1, solver="liblinear")` or `solver="saga"` | drives many weights to exactly 0: built-in feature selection |
| Elastic net | `l1_ratio=0.5, solver="saga"` | a mix of both |
| None | `C=np.inf` | pure maximum likelihood; weights blow up on separable data |

In [ ]:
l2_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000)).fit(Xc, yc)
l1_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, l1_ratio=1.0, solver="liblinear")).fit(Xc, yc)
nnz = lambda m: int(np.sum(np.abs(m[-1].coef_) > 1e-8))
print(f"non-zero coefficients out of 30: L2 {nnz(l2_model)}, L1 {nnz(l1_model)}")
assert nnz(l1_model) < nnz(l2_model)

coefs = l1_model[-1].coef_[0]
top = np.argsort(-np.abs(coefs))[:5]
print("\nstrongest L1 features (standardised units), odds ratio per +1 standard deviation:")
for j in top:
    print(f"  {data.feature_names[j]:<22} coef {coefs[j]:+.2f}   odds x{np.exp(coefs[j]):.2f}")

**Reading a coefficient:** with standardised features, $e^{w_j}$ is the factor the *odds* of malignancy get multiplied by for a one-standard-deviation increase in feature $j$, holding the others fixed. With correlated features (radius, perimeter and area here) the credit is split arbitrarily between them, so do not read these as causal.

## Try it yourself

**1.** Check the log-loss gradient $\frac1n X^\top(p-y)$ against finite differences at a random `w` on the blobs data (no penalty, no bias).

In [ ]:
# Solution — try it yourself first, then run this
w_test, eps = rng.normal(size=2), 1e-6
loss_fn = lambda w_: log_loss_np(yb, sigmoid(Xb @ w_))
numeric = np.array([(loss_fn(w_test + eps * e) - loss_fn(w_test - eps * e)) / (2 * eps) for e in np.eye(2)])
analytic = Xb.T @ (sigmoid(Xb @ w_test) - yb) / len(yb)
print("numeric :", numeric.round(6)); print("analytic:", analytic.round(6))
assert np.allclose(numeric, analytic, atol=1e-7)

**2.** Screening: using `p_oof` from the breast-cancer section, find the *highest* threshold that still catches at least 99% of malignant tumours. How many false alarms does that cost?

In [ ]:
# Solution — try it yourself first, then run this
ok = [t for t in thresholds if recall_score(yc, (p_oof >= t).astype(int)) >= 0.99]
t99 = max(ok)
pred99 = p_oof >= t99
print(f"threshold {t99:.2f}: recall {recall_score(yc, pred99.astype(int)):.3f}, "
      f"false alarms {np.sum((yc == 0) & pred99)} of {np.sum(yc == 0)} benign")

**3.** Imbalance. On a 95/5 dataset, compare minority-class recall for `LogisticRegression()` and `LogisticRegression(class_weight="balanced")`. What does `class_weight` do to the loss, and how is it related to moving the threshold?

In [ ]:
# Solution — try it yourself first, then run this
Xim, yim = make_classification(n_samples=4000, weights=[0.95], flip_y=0.02, class_sep=0.8, random_state=0)
Xim_tr, Xim_te, yim_tr, yim_te = train_test_split(Xim, yim, test_size=0.3, stratify=yim, random_state=0)
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw).fit(Xim_tr, yim_tr)
    print(f"class_weight={cw!s:>8}: minority recall {recall_score(yim_te, m.predict(Xim_te)):.3f}, "
          f"accuracy {m.score(Xim_te, yim_te):.3f}")
# 'balanced' multiplies each example's loss by n / (2 * n_class): minority mistakes cost ~10x more,
# which pushes the fitted probabilities (and so the effective 0.5 threshold) towards the minority class.

## Say it in an interview

- **30-second answer:** "Logistic regression computes a linear score, the logit, and squashes it with a sigmoid into P(y=1|x). It is trained by minimising log-loss, the Bernoulli negative log-likelihood, which is convex with gradient Xᵀ(p − y)/n. The boundary is linear in the features, so for curved boundaries you engineer features. For K classes you use a softmax over K logits: the same cross-entropy loss that trains LLMs over their vocabulary."
- **Why not MSE:** with a sigmoid it is non-convex, and its gradient vanishes exactly when the model is confidently wrong. Log-loss's gradient is p − y.
- **Threshold:** 0.5 is a default, not a law. With calibrated probabilities the cost-minimising threshold is C_FP / (C_FP + C_FN); a 10:1 miss cost gives about 0.09. Tune it on validation data. Routers and guardrail classifiers work the same way.
- **`C`:** inverse regularisation. Small `C` means a simpler, softer model; L1 zeroes weights (feature selection); L2 shrinks them. Scale features first, because the penalty treats all weights alike.
- **Softmax facts:** rows sum to 1, shift-invariant (subtract the max), two classes reduce to the sigmoid, and temperature z/T changes the peakedness, never the argmax.
- **Traps:** unscaled features with a penalty; separable data with no penalty (weights → ∞); reading correlated coefficients as causal; accuracy on imbalanced data; `penalty=` in new scikit-learn code (deprecated since 1.8, use `l1_ratio` / `C`).
- **Follow-ups:** "Is it a linear model?" (linear in log-odds). "How do you get probabilities from an SVM?" (calibration, [04](04_metrics_confusion_matrix_roc_pr.ipynb)). "One-vs-rest or multinomial?" (multinomial is the default for lbfgs; OvR trains K separate binary models whose scores do not naturally sum to 1).

## Next

- [04 · Metrics: confusion matrix, ROC and PR](04_metrics_confusion_matrix_roc_pr.ipynb): measuring what the threshold trades.
- [05 · Overfitting, regularisation and cross-validation](05_overfitting_regularization_cross_validation.ipynb): choosing `C` properly.
- [07 · kNN, SVM and Naive Bayes](07_knn_svm_naive_bayes.ipynb): other classifiers and their boundaries.
- [Decoding: temperature, top-k, top-p](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb) and [a neuron and backprop from scratch](../06_deep_learning/01_neuron_and_backprop_from_scratch.ipynb).
- [LiteLLM gateway and model routing](../16_production/06_litellm_gateway_and_model_routing.ipynb): the router, in production.
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapter 5), [genai_flow course](../../genai_flow/index.html), [interview bank](../../ai_interview_prep/index.html).